# 01 - Dataset Acquisition

This notebook verifies the acquisition and basic integrity of the datasets
used for the Crowd Violence / Riot Risk Detection project.

Datasets:
1. RWF-2000
2. Violent-Flows

Objectives:
- Verify dataset directories exist
- Count acquired videos
- Inspect dataset structure
- Check video file formats
- Verify that sample videos can be opened
- Record basic dataset statistics

No frame extraction, preprocessing, labeling, splitting, or model training
is performed in this notebook.

In [2]:
from pathlib import Path
import cv2

In [3]:
PROJECT_ROOT = Path("..")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

RWF_DIR = RAW_DATA_DIR / "rwf2000"
VIOLENTFLOWS_DIR = RAW_DATA_DIR / "violentflows"

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data:", RAW_DATA_DIR.resolve())
print("RWF-2000:", RWF_DIR.resolve())
print("Violent-Flows:", VIOLENTFLOWS_DIR.resolve())

Project root: C:\Users\shlok\riot detection\crowd-violence-ml
Raw data: C:\Users\shlok\riot detection\crowd-violence-ml\data\raw
RWF-2000: C:\Users\shlok\riot detection\crowd-violence-ml\data\raw\rwf2000
Violent-Flows: C:\Users\shlok\riot detection\crowd-violence-ml\data\raw\violentflows


In [6]:
paths = {
    "Raw data": RAW_DATA_DIR,
    "RWF-2000": RWF_DIR,
    "Violent-Flows": VIOLENTFLOWS_DIR,
}

for name, path in paths.items():
    print(f"{name}: {'✓ EXISTS' if path.exists() else '✗ MISSING'}")

Raw data: ✓ EXISTS
RWF-2000: ✓ EXISTS
Violent-Flows: ✓ EXISTS


In [7]:
print("RWF-2000 structure:\n")

for path in RWF_DIR.rglob("*"):
    if path.is_dir():
        print("[DIR] ", path.relative_to(RWF_DIR))

RWF-2000 structure:

[DIR]  train
[DIR]  val
[DIR]  train\Train_Fight
[DIR]  train\Train_NonFight
[DIR]  val\Val_Fight
[DIR]  val\Val_NonFight


In [8]:
video_extensions = {".avi", ".mp4", ".mov", ".mkv", ".webm"}

rwf_videos = [
    path for path in RWF_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in video_extensions
]

print("Total RWF-2000 videos:", len(rwf_videos))

Total RWF-2000 videos: 2000


In [9]:
rwf_counts = {
    "Train/Fight": len([
        p for p in (RWF_DIR / "train" / "Train_Fight").iterdir()
        if p.is_file()
    ]),
    
    "Train/NonFight": len([
        p for p in (RWF_DIR / "train" / "Train_NonFight").iterdir()
        if p.is_file()
    ]),
    
    "Val/Fight": len([
        p for p in (RWF_DIR / "val" / "Val_Fight").iterdir()
        if p.is_file()
    ]),
    
    "Val/NonFight": len([
        p for p in (RWF_DIR / "val" / "Val_NonFight").iterdir()
        if p.is_file()
    ])
}

for split, count in rwf_counts.items():
    print(f"{split}: {count}")

print("\nTotal:", sum(rwf_counts.values()))

Train/Fight: 800
Train/NonFight: 800
Val/Fight: 200
Val/NonFight: 200

Total: 2000


In [10]:
extension_counts = {}

for path in rwf_videos:
    extension = path.suffix.lower()
    extension_counts[extension] = extension_counts.get(extension, 0) + 1

print(extension_counts)

{'.avi': 2000}


In [11]:
def check_video(video_path):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return False

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    cap.release()

    return frame_count > 0 and fps > 0

In [12]:
print("Testing 10 RWF-2000 videos:\n")

for video in rwf_videos[:10]:
    status = check_video(video)
    print(f"{video.name}: {'OK' if status else 'FAILED'}")

Testing 10 RWF-2000 videos:

-1l5631l3fg_0.avi: OK
-1l5631l3fg_1.avi: OK
-1l5631l3fg_2.avi: OK
0H2s9UJcNJ0_0.avi: OK
0H2s9UJcNJ0_2.avi: OK
0H2s9UJcNJ0_3.avi: OK
0H2s9UJcNJ0_4.avi: OK
0H2s9UJcNJ0_5.avi: OK
0lHQ2f0d_0.avi: OK
0lHQ2f0d_1.avi: OK


In [13]:
print("Violent-Flows structure:\n")

for path in VIOLENTFLOWS_DIR.rglob("*"):
    if path.is_dir():
        print("[DIR] ", path.relative_to(VIOLENTFLOWS_DIR))

Violent-Flows structure:

[DIR]  movies
[DIR]  movies\1
[DIR]  movies\2
[DIR]  movies\3
[DIR]  movies\4
[DIR]  movies\5
[DIR]  movies\1\NonViolence
[DIR]  movies\1\Violence
[DIR]  movies\2\NonViolence
[DIR]  movies\2\Violence
[DIR]  movies\3\NonViolence
[DIR]  movies\3\Violence
[DIR]  movies\4\NonViolence
[DIR]  movies\4\Violence
[DIR]  movies\5\NonViolence
[DIR]  movies\5\Violence


In [14]:
vf_videos = [
    path for path in VIOLENTFLOWS_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in video_extensions
]

print("Total Violent-Flows videos:", len(vf_videos))

Total Violent-Flows videos: 246


In [15]:
vf_counts = {}

for group in ["1", "2", "3", "4", "5"]:
    violence_dir = VIOLENTFLOWS_DIR / "movies" / group / "Violence"
    nonviolence_dir = VIOLENTFLOWS_DIR / "movies" / group / "NonViolence"

    vf_counts[f"{group}/Violence"] = len([
        p for p in violence_dir.iterdir() if p.is_file()
    ])

    vf_counts[f"{group}/NonViolence"] = len([
        p for p in nonviolence_dir.iterdir() if p.is_file()
    ])

for category, count in vf_counts.items():
    print(f"{category}: {count}")

print("\nTotal:", sum(vf_counts.values()))

1/Violence: 25
1/NonViolence: 25
2/Violence: 25
2/NonViolence: 25
3/Violence: 25
3/NonViolence: 25
4/Violence: 24
4/NonViolence: 24
5/Violence: 24
5/NonViolence: 24

Total: 246


In [16]:
violence_csv = VIOLENTFLOWS_DIR / "Violence.csv"
nonviolence_csv = VIOLENTFLOWS_DIR / "NonViolence.csv"

print("Violence.csv:", "✓ EXISTS" if violence_csv.exists() else "✗ MISSING")
print("NonViolence.csv:", "✓ EXISTS" if nonviolence_csv.exists() else "✗ MISSING")

Violence.csv: ✓ EXISTS
NonViolence.csv: ✓ EXISTS


In [17]:
print("Testing 10 Violent-Flows videos:\n")

for video in vf_videos[:10]:
    status = check_video(video)
    print(f"{video.name}: {'OK' if status else 'FAILED'}")

Testing 10 Violent-Flows videos:

football_crowds_cheering__Andres_Iniesta_Standing_Ovations__allas11__WKv7pmjgDeg.avi: OK
football_crowds_cheering__Arkansas_vs_ULM_September_11_2010_Woo_Pig_Sooie__jayd243__RfMRWT.avi: OK
football_crowds_cheering__Australia_vs_Ghana__cubicspacedivision__hy9sgs_Rzbg.avi: OK
football_crowds_cheering__Cal_Beats_Stanford_Big_Game_2002_short_low_res_version__belmonto.avi: OK
football_crowds_cheering__Chang_Rai_United_Football_Match__goldfly___aTVnHfqClA.avi: OK
football_crowds_cheering__FIFA_2010_USA_v_ENG_Dupont_Cirlce_Washington_DC__icallmyselfbria.avi: OK
football_crowds_cheering__football_game__banonit123__m5pMOoPH21w.avi: OK
football_crowds_cheering__Varsity_Football_Cardinal_O_Hara_vs_Roman_Catholic__Varsity365__.avi: OK
football_crowds_cheering__Welcome_Home_ACC_Champions_part_A__jamesllvt__UGurBSFE8Es.avi: OK
football_crowds_cheering__World_Cup_2010_Korea_vs_Uruguay_Han_River_Seoul_Pre_game__vivach.avi: OK
